In [1]:
# ============================================================
# ПРАКТИКА 15
# Критерії узгодженості й таблиці спряженості: χ²
# Варіант 4 — Харків
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, chisquare


# ============================================================
# ПОБУДОВА ВИХІДНОГО НАБОРУ ДАНИХ
# ============================================================

np.random.seed(42)

base_temp = 8.8       # середньорічна температура Харкова
amplitude = 15        # сезонна амплітуда
city = "Харків"


def season(month):
    if month in (12, 1, 2):
        return "зима"
    if month in (3, 4, 5):
        return "весна"
    if month in (6, 7, 8):
        return "літо"
    return "осінь"


rows = []

for year in [2021, 2022, 2023, 2024]:
    for month in range(1, 13):

        seasonal = amplitude * np.cos(
            (month - 7) / 12 * 2 * np.pi
        )

        noise = np.random.normal(0, 1.0)

        temp = round(
            base_temp + seasonal + noise,
            1
        )

        diff = temp - base_temp

        if diff < -3:
            norm_cat = "холодніше"
        elif diff > 3:
            norm_cat = "тепліше"
        else:
            norm_cat = "звичайно"

        rows.append({
            "місто": city,
            "рік": year,
            "місяць": month,
            "температура": temp,
            "сезон": season(month),
            "відхилення_від_норми": norm_cat
        })


climate = pd.DataFrame(rows)

print("Вихідний набір даних:")
print(climate)


# ============================================================
# ЗАВДАННЯ 1. Таблиця спряженості
# ============================================================

print("\n")
print("ЗАВДАННЯ 1. Таблиця спряженості")
print("-" * 60)

table = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"]
)

print("Таблиця спряженості:")
print(table)


# Нормована таблиця — частки всередині кожного сезону
table_normalized = pd.crosstab(
    climate["сезон"],
    climate["відхилення_від_норми"],
    normalize="index"
)

print("\nНормована таблиця:")
print(table_normalized)


# Знаходимо клітинку з найбільшою кількістю спостережень
max_cell = table.stack().idxmax()
max_value = table.stack().max()

print(
    f"\nНайчастіша комбінація: сезон = '{max_cell[0]}', "
    f"відхилення = '{max_cell[1]}' ({max_value} спостережень)"
)


# ============================================================
# ЗАВДАННЯ 2. Критерій незалежності χ²
# ============================================================

print("\n")
print("ЗАВДАННЯ 2. Критерій незалежності χ²")
print("-" * 60)

chi2, p_value, dof, expected = chi2_contingency(table)

print("χ² =", chi2)
print("p-value =", p_value)
print("Ступені вільності =", dof)

expected_table = pd.DataFrame(
    expected,
    index=table.index,
    columns=table.columns
)

print("\nОчікувані частоти:")
print(expected_table.round(2))

alpha = 0.05

if p_value < alpha:
    print("\nРішення: відхиляємо H0.")
    print(
        "Висновок: існують статистично значущі підстави "
        "вважати, що сезон і відхилення від норми пов'язані."
    )
else:
    print("\nРішення: не відхиляємо H0.")
    print(
        "Висновок: недостатньо статистичних підстав "
        "стверджувати, що сезон і відхилення від норми пов'язані."
    )


# Порівняння observed та expected
difference = abs(table - expected_table)

max_difference_cell = difference.stack().idxmax()
max_difference = difference.stack().max()

print(
    f"\nНайбільша абсолютна розбіжність між observed та expected: "
    f"'{max_difference_cell[0]}' + '{max_difference_cell[1]}'"
)
print("Розбіжність =", max_difference)


# ============================================================
# ЗАВДАННЯ 3. Перевірка умови застосовності
# ============================================================

print("\n")
print("ЗАВДАННЯ 3. Перевірка умови застосовності")
print("-" * 60)

print("Матриця очікуваних частот:")
print(expected_table.round(2))

# Знаходимо всі клітинки з expected < 5
low_expected = expected_table < 5

print("\nКлітинки з очікуваною частотою < 5:")
print(expected_table[low_expected])

if (expected < 5).any():
    print(
        "\nУмова застосовності порушена: "
        "є очікувані частоти менші за 5."
    )

    print(
        "Для виправлення можна об'єднати категорії "
        "'холодніше' та 'звичайно' в категорію 'не тепліше'."
    )

    climate["відхилення_об'єднане"] = np.where(
        climate["відхилення_від_норми"] == "тепліше",
        "тепліше",
        "не тепліше"
    )

    table_combined = pd.crosstab(
        climate["сезон"],
        climate["відхилення_об'єднане"]
    )

    print("\nНова таблиця після об'єднання:")
    print(table_combined)

    chi2_combined, p_combined, dof_combined, expected_combined = \
        chi2_contingency(table_combined)

    print("\nПовторний критерій χ²:")
    print("χ² =", chi2_combined)
    print("p-value =", p_combined)

else:
    print(
        "\nУмова застосовності виконується: "
        "усі очікувані частоти >= 5."
    )


# ============================================================
# ЗАВДАННЯ 4. Критерій узгодженості для сезонів
# ============================================================

print("\n")
print("ЗАВДАННЯ 4. Критерій узгодженості для сезонів")
print("-" * 60)

season_counts = climate["сезон"].value_counts()

# Впорядковуємо сезони
season_counts = season_counts.reindex([
    "зима",
    "весна",
    "літо",
    "осінь"
])

print("Фактичні частоти сезонів:")
print(season_counts)

n = len(climate)

# Кожен сезон має становити 25%
expected_seasons = [
    0.25 * n,
    0.25 * n,
    0.25 * n,
    0.25 * n
]

chi2_seasons, p_seasons = chisquare(
    f_obs=season_counts,
    f_exp=expected_seasons
)

print("\nОчікувані частоти:")
print(expected_seasons)

print("\nχ² =", chi2_seasons)
print("p-value =", p_seasons)

if p_seasons < alpha:
    print(
        "\nВисновок: розподіл сезонів статистично "
        "значуще відрізняється від рівномірного 25%/25%/25%/25%."
    )
else:
    print(
        "\nВисновок: немає достатніх статистичних підстав "
        "стверджувати, що розподіл сезонів відрізняється "
        "від рівномірного 25%/25%/25%/25%."
    )


# ============================================================
# ЗАВДАННЯ 5. Власна заявлена пропорція
# ============================================================

print("\n")
print("ЗАВДАННЯ 5. Власна заявлена пропорція")
print("-" * 60)

# H0:
# 20% — холодніше
# 60% — звичайно
# 20% — тепліше

categories = [
    "холодніше",
    "звичайно",
    "тепліше"
]

observed_counts = (
    climate["відхилення_від_норми"]
    .value_counts()
    .reindex(categories)
)

# Наша заявлена пропорція
expected_proportions = [0.20, 0.60, 0.20]

expected_counts = [
    p * n for p in expected_proportions
]

print("Фактичні частоти:")
print(observed_counts)

print("\nЗаявлені очікувані частоти:")
print(expected_counts)

chi2_prop, p_prop = chisquare(
    f_obs=observed_counts,
    f_exp=expected_counts
)

print("\nχ² =", chi2_prop)
print("p-value =", p_prop)

if p_prop < alpha:
    print(
        "\nРішення: відхиляємо H0."
    )
    print(
        "Висновок: фактичний розподіл категорій "
        "статистично значуще відрізняється "
        "від заявленої пропорції 20%/60%/20%."
    )
else:
    print(
        "\nРішення: не відхиляємо H0."
    )
    print(
        "Висновок: немає достатніх статистичних підстав "
        "стверджувати, що фактичний розподіл відрізняється "
        "від заявленої пропорції 20%/60%/20%."
    )


Вихідний набір даних:
     місто   рік  місяць  температура  сезон відхилення_від_норми
0   Харків  2021       1         -5.7   зима            холодніше
1   Харків  2021       2         -4.3   зима            холодніше
2   Харків  2021       3          1.9  весна            холодніше
3   Харків  2021       4         10.3  весна             звичайно
4   Харків  2021       5         16.1  весна              тепліше
5   Харків  2021       6         21.6   літо              тепліше
6   Харків  2021       7         25.4   літо              тепліше
7   Харків  2021       8         22.6   літо              тепліше
8   Харків  2021       9         15.8  осінь              тепліше
9   Харків  2021      10          9.3  осінь             звичайно
10  Харків  2021      11          0.8  осінь            холодніше
11  Харків  2021      12         -4.7   зима            холодніше
12  Харків  2022       1         -6.0   зима            холодніше
13  Харків  2022       2         -6.1   зима          

Контрольні питання
1. Чому в статистиці χ² використовується квадрат відхилення і ділення на очікувану частоту?
Формула χ² порівнює фактичні та очікувані частоти:

χ
2
=
∑
(
O
−
E
)
2
E

Квадрат потрібен, щоб додатні та від'ємні відхилення не компенсували одне одного. Ділення на очікувану частоту враховує масштаб: однакова абсолютна різниця має різну важливість для великих і малих очікуваних частот.

2. Як обчислюється очікувана частота клітинки?
Для таблиці спряженості очікувана частота обчислюється так:

E
=
сума рядка
×
сума стовпця
загальна сума

Ця формула показує, скільки спостережень ми очікували б отримати в клітинці, якби дві категоріальні змінні були незалежними.

3. Чим відрізняється «p-value велике — немає підстав відхилити H0» від «доведено, що змінні незалежні»?
Велике p-value означає, що отриманих даних недостатньо для відхилення H0 про незалежність змінних. Це не доводить, що змінні незалежні. Відсутність статистично значущого зв'язку не є доказом повної відсутності зв'язку.

4. Що робити, якщо очікувана частота менша за 5?
Якщо очікувані частоти занадто малі, наближення, на якому ґрунтується критерій χ², може бути ненадійним. Можна об'єднати близькі категорії, щоб збільшити очікувані частоти, а потім повторити критерій. У нашому завданні, наприклад, можна об'єднати категорії «холодніше» та «звичайно» в категорію «не тепліше».